# VoiceLens — Whisper QLoRA fine-tuning (Colab)

Fine-tunes `whisper-small` on Indian-English (Common Voice 17) with **QLoRA** and prints a **WER before/after** comparison.

**Before you start:**
1. Runtime → Change runtime type → **T4 GPU**.
2. Create a free HuggingFace account, open the [Common Voice 17 dataset page](https://huggingface.co/datasets/mozilla-foundation/common_voice_17_0) and click **Agree to access**.
3. Create an HF access token (Settings → Access Tokens) — you'll paste it below.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install dependencies

In [ ]:
# Pure-Python ML libs: pinned for reproducibility.
!pip install -q \
  "transformers==4.46.3" \
  "peft==0.13.2" \
  "accelerate>=1.1.1" \
  "datasets==2.20.0" \
  "huggingface_hub==0.24.7" \
  "evaluate==0.4.3" \
  "jiwer==3.0.5" \
  "librosa==0.10.2.post1" \
  "soundfile==0.12.1" \
  "pyyaml"

# bitsandbytes: FORCE the latest (-U). Older builds lack the cuda128 binary and
# import the removed `triton.ops` module, so they crash on Colab's CUDA/Triton.
# A plain `>=0.45.0` won't upgrade an already-installed 0.45.0 — `-U` does.
!pip install -q -U bitsandbytes

!pip show bitsandbytes | grep -i version

## 3. Get the VoiceLens fine-tuning scripts

Replace the URL with your repo, **or** skip this cell and upload the `finetune/` folder via the file browser on the left.

In [ ]:
REPO_URL = "https://github.com/<your-username>/VoiceLens.git"  # <-- edit me
!git clone $REPO_URL voicelens
%cd voicelens

## 4. Authenticate with HuggingFace (gated dataset)

In [ ]:
import os
from getpass import getpass
os.environ["HF_TOKEN"] = getpass("Paste your HuggingFace token: ")

## 5. Prepare the dataset (stream + filter Indian-English, 16 kHz log-mels)

In [ ]:
!python finetune/prepare_dataset.py

## 6. Train (QLoRA)

Watch for the **trainable-parameters** print near the top — it should show well under 1% of the model is being trained.

In [ ]:
!python finetune/train.py

## 7. Evaluate — WER before vs after

In [ ]:
!python finetune/evaluate.py

## 8. Download the adapter

The adapter weights live in `finetune/output/adapter/` (only a few MB). Download them and drop them into the same path in your local checkout so `backend/transcriber.py` picks them up.

In [ ]:
import shutil
shutil.make_archive("adapter", "zip", "finetune/output/adapter")
from google.colab import files
files.download("adapter.zip")